# Gaussian Process Isopleth Surface Builder

This notebook mirrors the workflow of `RBF_Isopleth_Surface.ipynb` but fits a Gaussian Process Regression (GPR) model to create a smooth surface from isopleth samples. Edit the configuration cell, then run the sections top-to-bottom to refresh the fit, diagnostics, and plots.

## Quick Start
1. Update the `CONFIG` dictionary (data path, column names, parameter to model, GPR hyperparameters).
2. Run *Load & Prepare Data* to get clean predictor/response arrays.
3. Execute *Fit Gaussian Process Surface* to build the interpolated grid.
4. Check *K-Fold Evaluation* for cross-validated quality metrics.
5. Render the 2D contour, static 3D, and interactive 3D plots to inspect the surface.

In [ ]:
from pathlib import Path

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import ticker, colors
from IPython.display import display
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel, WhiteKernel
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
CONFIG = {
    # Data + column settings -------------------------------------------------
    'data_path': Path('../data/utah/filtered_by_clearing_index/SR_clearing_nonmax_summer_1000.csv'),
    'lon_col': 'VOC',
    'lat_col': 'NOx',
    'value_col': 'Ozone',  # Change this to switch which parameter is modeled.
    'filters': {
        # Example: 'parameter': ['Carbon disulfide']
    },
    'save': False,
    'output_dir': Path('../plots/utah/SR_filtered/Clearing_Index_Filtered/1000/No_max/GPR'),
    # Surface + evaluation settings -----------------------------------------
    'grid_resolution': 80,
    'kfold': {
        'splits': 5,
        'shuffle': True,
        'random_state': 42,
    },
    # Gaussian Process hyperparameters --------------------------------------
    'gpr': {
        'alpha': 0.3,
        'normalize_y': True,
        # More restarts so the internal likelihood optimizer can explore.
        'n_restarts_optimizer': 10,
        'kernel': {
            'type': 'RationalQuadratic',  # Options: 'RBF', 'Matern', 'RationalQuadratic'
            'length_scale': 3.0,
            'length_scale_bounds': (1e-8, 1e8),
            'nu': 1.5,
            'constant': 1.0,
            'constant_bounds': (1e-5, 1e8),
            'white_noise': 0.05,
            'white_noise_bounds': (1e-8, 10.0),
            'alpha_bounds': (1e-5, 1e8),
            'alpha': 1.0,
        },
    },
    # Grid-search + parallelization controls --------------------------------
    'grid_search': {
        'parallel': {
            # -1 will use all available CPU cores; set to an int to cap parallel jobs.
            'n_jobs': -1,
            # Switch to True for threading; leave False to prefer process-based workers.
            'prefer_threads': False,
            'batch_size': 'auto',
            'verbose': 0,
        },
    },
    # Plot styling -----------------------------------------------------------
    'plot': {
        'cmap': 'viridis',
        'contour_levels': 50,
        'colorbar_limits': (30, 85),
        'colorbar_step': 5,
        'point_size': 35,
        'surface_alpha': 0.85,
        'scatter_color': 'firebrick',
        'plotly_colorscale': 'Viridis',
    },
    'title': 'Gaussian Process Surface - Non Max All Time Clearing index under 1000',
}



In [ ]:
def build_kernel(kernel_cfg):
    kernel_cfg = kernel_cfg or {}
    kind = (kernel_cfg.get('type') or 'RBF').lower()
    length_scale = kernel_cfg.get('length_scale', 1.0)
    bounds = kernel_cfg.get('length_scale_bounds', (1e-5, 1e5))

    if kind == 'matern':
        base = Matern(
            length_scale=length_scale,
            length_scale_bounds=bounds,
            nu=kernel_cfg.get('nu', 1.5),
        )
    elif kind == 'rationalquadratic':
        base = RationalQuadratic(
            length_scale=length_scale,
            alpha=kernel_cfg.get('alpha', 1.0),
            length_scale_bounds=bounds,
            alpha_bounds=kernel_cfg.get('alpha_bounds', (1e-5, 1e5)),
        )
    else:
        base = RBF(length_scale=length_scale, length_scale_bounds=bounds)

    constant = kernel_cfg.get('constant')
    if constant is not None:
        const_bounds = kernel_cfg.get('constant_bounds', (1e-5, 1e5))
        base = ConstantKernel(constant_value=constant, constant_value_bounds=const_bounds) * base

    noise = kernel_cfg.get('white_noise')
    if noise is not None and noise > 0:
        noise_bounds = kernel_cfg.get('white_noise_bounds', (1e-5, 1e5))
        base += WhiteKernel(noise_level=noise, noise_level_bounds=noise_bounds)

    return base


In [ ]:
def make_gpr(gpr_cfg):
    gpr_cfg = dict(gpr_cfg)
    kernel_cfg = gpr_cfg.pop('kernel', {})
    kernel = build_kernel(kernel_cfg)
    return GaussianProcessRegressor(kernel=kernel, **gpr_cfg)


def evaluate_gpr_kfold(lon_array, lat_array, value_array, gpr_cfg, kfold_cfg):
    lon_array = np.asarray(lon_array, dtype=float)
    lat_array = np.asarray(lat_array, dtype=float)
    value_array = np.asarray(value_array, dtype=float)

    predictors = np.column_stack((lon_array, lat_array))
    kf = KFold(
        n_splits=kfold_cfg['splits'],
        shuffle=kfold_cfg.get('shuffle', False),
        random_state=kfold_cfg.get('random_state'),
    )

    fold_rows = []
    for fold_id, (train_idx, test_idx) in enumerate(kf.split(predictors), start=1):
        gpr = make_gpr(gpr_cfg)
        gpr.fit(predictors[train_idx], value_array[train_idx])
        preds = gpr.predict(predictors[test_idx])

        mse = mean_squared_error(value_array[test_idx], preds)
        rmse = float(np.sqrt(mse))

        fold_rows.append({
            'fold': fold_id,
            'n_train': len(train_idx),
            'n_test': len(test_idx),
            'rmse': rmse,
            'mae': mean_absolute_error(value_array[test_idx], preds),
            'r2': r2_score(value_array[test_idx], preds),
        })

    return pd.DataFrame(fold_rows)


In [ ]:
# Load & Prepare Data
cfg = CONFIG
path = cfg['data_path']
lat_col = cfg['lat_col']
lon_col = cfg['lon_col']
value_col = cfg['value_col']

SAVE_OUTPUTS = bool(cfg.get('save', False))
output_dir_cfg = cfg.get('output_dir') or (path.parent / 'gpr_outputs')
OUTPUT_DIR = Path(output_dir_cfg).expanduser()
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = Path.cwd() / OUTPUT_DIR
RUN_ID = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_PREFIX = f'gpr_surface_{RUN_ID}'

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


if not path.exists():
    raise FileNotFoundError(f'Data file not found: {path}')

raw = pd.read_csv(path)

for col, allowed in cfg['filters'].items():
    if allowed:
        raw = raw[raw[col].isin(allowed)]

required_surface_cols = ['Ozone', 'VOC', 'NOx']
missing_surface_cols = [col for col in required_surface_cols if col not in raw.columns]
if missing_surface_cols:
    raise ValueError(f"Missing required columns for surface generation: {missing_surface_cols}")
raw_before = len(raw)
raw = raw.dropna(subset=required_surface_cols).copy()
surface_rows_dropped = raw_before - len(raw)
if surface_rows_dropped:
    print(f"Removed {surface_rows_dropped} rows with null surface inputs (Ozone/VOC/NOx).")

needed_cols = [lon_col, lat_col, value_col]
df = raw[needed_cols].copy()
for col in needed_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=needed_cols).reset_index(drop=True)
print(f'Samples available: {len(df)}')
df.head()

In [ ]:
# Fit Gaussian Process Surface
lon = df[lon_col].to_numpy()
lat = df[lat_col].to_numpy()
values = df[value_col].to_numpy()

predictors = np.column_stack((lon, lat))
gpr = make_gpr(cfg['gpr'])
gpr.fit(predictors, values)

res = cfg['grid_resolution']
lon_lin = np.linspace(lon.min(), lon.max(), res)
lat_lin = np.linspace(lat.min(), lat.max(), res)
lon_grid, lat_grid = np.meshgrid(lon_lin, lat_lin)

surface_inputs = np.column_stack((lon_grid.ravel(), lat_grid.ravel()))
value_pred, std_pred = gpr.predict(surface_inputs, return_std=True)
value_grid = value_pred.reshape(lon_grid.shape)
std_grid = std_pred.reshape(lon_grid.shape)
variance_grid = np.square(std_grid)

surface_summary = {
    'value_min': float(value_grid.min()),
    'value_max': float(value_grid.max()),
    'value_mean': float(value_grid.mean()),
    'variance_max': float(variance_grid.max()),
    'variance_mean': float(variance_grid.mean()),
}

GRID_VALUE_MAX = float(surface_summary['value_max'])

metadata_record = {
    'run_id': RUN_ID,
    'data_path': str(path),
    'lon_col': lon_col,
    'lat_col': lat_col,
    'value_col': value_col,
    'grid_resolution': res,
    'filters': json.dumps(cfg.get('filters', {})),
    'value_min': surface_summary['value_min'],
    'value_max': surface_summary['value_max'],
    'value_mean': surface_summary['value_mean'],
    'plot_title': cfg.get('title', ''),
}

kfold_cfg = dict(cfg.get('kfold', {}))
for key, value in kfold_cfg.items():
    metadata_record[f'kfold_{key}'] = value

gpr_cfg = dict(cfg.get('gpr', {}))
for key, value in gpr_cfg.items():
    if key == 'kernel' and isinstance(value, dict):
        for kernel_key, kernel_value in value.items():
            metadata_record[f'gpr_kernel_{kernel_key}'] = kernel_value
    else:
        metadata_record[f'gpr_{key}'] = value

for key, value in cfg.get('plot', {}).items():
    metadata_record[f'plot_{key}'] = value

if SAVE_OUTPUTS:
    params_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_params.csv'
    pd.DataFrame([metadata_record]).to_csv(params_path, index=False)

surface_summary


In [ ]:
# K-Fold Evaluation
kfold_cfg = cfg['kfold']
eval_df = evaluate_gpr_kfold(lon, lat, values, cfg['gpr'], kfold_cfg)
summary = eval_df.agg({'rmse': ['mean', 'std'], 'mae': ['mean', 'std'], 'r2': ['mean', 'std']})
print('Fold metrics:')
display(eval_df)
print('Aggregated metrics:')
display(summary)


In [ ]:
# 2D Contour Plot
plot_cfg = cfg['plot']
plot_title = cfg.get('title') or 'Gaussian Process Surface - 2D Contours'
fig, ax = plt.subplots(figsize=(8, 6))

value_max = float(surface_summary['value_max'])
if not np.isfinite(value_max) or value_max <= 0:
    value_max = 1.0

colorbar_limits = plot_cfg.get('colorbar_limits')
if colorbar_limits and len(colorbar_limits) == 2:
    vmin, vmax = map(float, colorbar_limits)
else:
    vmin, vmax = 0.0, value_max
norm = colors.Normalize(vmin=vmin, vmax=vmax)

level_step = plot_cfg.get('colorbar_step')
levels_cfg = plot_cfg.get('contour_levels', 15)
use_step = level_step is not None and float(level_step) > 0
if use_step:
    step = float(level_step)
    levels = np.arange(norm.vmin, norm.vmax + step, step)
elif isinstance(levels_cfg, int):
    levels = np.linspace(norm.vmin, norm.vmax, levels_cfg)
else:
    levels = levels_cfg

contour = ax.contourf(lon_grid, lat_grid, value_grid, levels=levels, cmap=plot_cfg['cmap'], norm=norm)
scatter = ax.scatter(
    lon,
    lat,
    c=values,
    cmap=plot_cfg['cmap'],
    edgecolor='white',
    linewidth=0.5,
    s=plot_cfg['point_size'],
    norm=norm,
)

cbar = fig.colorbar(contour, ax=ax, label=value_col)
if use_step:
    cbar.set_ticks(np.arange(norm.vmin, norm.vmax + step, step))
ax.set_xlabel(lon_col)
ax.set_ylabel(lat_col)
#ax.set_xlim(0, 120)
ax.set_title(plot_title)
ax.legend([scatter], ['Observed points'], loc='best')
ax.xaxis.set_major_locator(ticker.MaxNLocator(6))
ax.yaxis.set_major_locator(ticker.MaxNLocator(6))

if SAVE_OUTPUTS:
    contour_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_2d_contour.png'
    fig.savefig(contour_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Predictive Variance Contour Plot
plot_cfg = cfg['plot']
plot_title = (cfg.get('title') or 'Gaussian Process Surface') + ' - Uncertainty for 95% confidence interval $2\sigma$'
fig, ax = plt.subplots(figsize=(8, 6))

variance_max = float(np.nanmax(variance_grid))
variance_max = 25
if not np.isfinite(variance_max) or variance_max <= 0:
    variance_max = 1.0
norm = colors.Normalize(vmin=10, vmax=variance_max)

levels_cfg = plot_cfg['contour_levels']
if isinstance(levels_cfg, int):
    levels = np.linspace(norm.vmin, norm.vmax, levels_cfg)
else:
    levels = levels_cfg

contour = ax.contourf(lon_grid, lat_grid, np.sqrt(variance_grid)*2, levels=levels, cmap=plot_cfg['cmap'], norm=norm)
scatter = ax.scatter(
    lon,
    lat,
    c='white',
    edgecolor='black',
    linewidth=0.5,
    s=max(plot_cfg['point_size'] / 1.5, 10),
    alpha=0.85,
)

cbar = fig.colorbar(contour, ax=ax, label='95% confidence interval $2\sigma$')
ax.set_xlabel(lon_col)
ax.set_ylabel(lat_col)
ax.set_xlabel(lon_col)
ax.set_xlabel(lon_col)
ax.set_ylabel(lat_col)
ax.set_xlim(0, 120)
ax.set_title(plot_title)
ax.set_title(plot_title)
ax.legend([scatter], ['Observed points'], loc='best')
ax.xaxis.set_major_locator(ticker.MaxNLocator(6))
ax.yaxis.set_major_locator(ticker.MaxNLocator(6))

if SAVE_OUTPUTS:
    variance_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_variance_contour.png'
    fig.savefig(variance_path, dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
# Static 3D Surface Plot
plot_cfg = cfg['plot']
plot_title = cfg.get('title') or 'Gaussian Process Surface - 3D Mesh'
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 - activates 3D projection

value_max = float(surface_summary['value_max'])
if not np.isfinite(value_max) or value_max <= 0:
    value_max = 1.0

colorbar_limits = plot_cfg.get('colorbar_limits')
if colorbar_limits and len(colorbar_limits) == 2:
    vmin, vmax = map(float, colorbar_limits)
else:
    vmin, vmax = 0.0, value_max
norm = colors.Normalize(vmin=vmin, vmax=vmax)

level_step = plot_cfg.get('colorbar_step')
use_step = level_step is not None and float(level_step) > 0
step = float(level_step) if use_step else None

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(
    lon_grid,
    lat_grid,
    value_grid,
    cmap=plot_cfg['cmap'],
    norm=norm,
    alpha=plot_cfg['surface_alpha'],
    linewidth=0,
    antialiased=True,
)
ax.scatter(lon, lat, values, color=plot_cfg['scatter_color'], s=plot_cfg['point_size'], depthshade=False)

ax.set_xlabel(lon_col)
ax.set_ylabel(lat_col)
ax.set_zlabel(value_col)
ax.set_title(plot_title)
cbar = fig.colorbar(surface, ax=ax, shrink=0.6, label=value_col)
if use_step:
    cbar.set_ticks(np.arange(norm.vmin, norm.vmax + step, step))

if SAVE_OUTPUTS:
    surface_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_3d_surface.png'
    fig.savefig(surface_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Interactive 3D Plot (Plotly)
try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError('Plotly is required for the interactive 3D view. Install it with `pip install plotly`.') from exc

plotly_scale = plot_cfg.get('plotly_colorscale') or plot_cfg['cmap'].capitalize()
plot_title = cfg.get('title') or 'Gaussian Process Surface - Interactive 3D'
marker_size = max(plot_cfg['point_size'] / 4, 4)
value_max = float(surface_summary['value_max'])
if not np.isfinite(value_max) or value_max <= 0:
    value_max = 1.0

fig = go.Figure()
fig.add_surface(
    x=lon_grid,
    y=lat_grid,
    z=value_grid,
    colorscale=plotly_scale,
    opacity=plot_cfg['surface_alpha'],
    showscale=True,
    name='GPR surface',
    cmin=0.0,
    cmax=value_max,
)
fig.add_scatter3d(
    x=lon,
    y=lat,
    z=values,
    mode='markers',
    marker=dict(
        size=marker_size,
        color=values,
        colorscale=plotly_scale,
        line=dict(width=0.5, color='gray'),
        cmin=0.0,
        cmax=value_max,
    ),
    name='Observed points',
)
fig.update_layout(
    width=900,
    height=650,
    scene=dict(
        xaxis_title=lon_col,
        yaxis_title=lat_col,
        zaxis_title=value_col,
    ),
    title=plot_title,
)
fig.show()

if SAVE_OUTPUTS:
    html_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_interactive_surface.html'
    fig.write_html(html_path)


In [ ]:
# Progressive Hyperparameter Grid Search (RMSE minimization)
from itertools import product
from functools import partial
import copy

def _unique_preserve(values):
    seen = set()
    ordered = []
    for item in values:
        if item not in seen:
            ordered.append(item)
            seen.add(item)
    return ordered

def _apply_gpr_updates(base_cfg, updates):
    candidate = copy.deepcopy(base_cfg)
    kernel_updates = {}
    for key, value in updates.items():
        if key.startswith('kernel.'):
            kernel_updates[key.split('.', 1)[1]] = value
        else:
            candidate[key] = value
    kernel = dict(candidate.get('kernel', {}))
    kernel.update(kernel_updates)
    candidate['kernel'] = kernel
    return candidate

def _evaluate_combo(combo, param_names, lon_array, lat_array, value_array, base_gpr_cfg, cv_cfg):
    updates = dict(zip(param_names, combo))
    candidate_cfg = _apply_gpr_updates(base_gpr_cfg, updates)
    try:
        fold_df = evaluate_gpr_kfold(
            lon_array,
            lat_array,
            value_array,
            candidate_cfg,
            cv_cfg,
        )
        return {
            **updates,
            'rmse_mean': fold_df['rmse'].mean(),
            'mae_mean': fold_df['mae'].mean(),
            'r2_mean': fold_df['r2'].mean(),
            'status': 'ok',
        }
    except Exception as exc:
        return {
            **updates,
            'rmse_mean': np.inf,
            'mae_mean': np.inf,
            'r2_mean': np.nan,
            'status': f'failed: {exc}',
        }

def _run_parallel_or_sequential(jobs, fn, items, prefer_threads=False, batch_size='auto', verbose=0):
    if jobs in (None, 0, 1):
        return [fn(item) for item in items]

    try:
        from joblib import Parallel, delayed, cpu_count
    except Exception as exc:
        print(f'joblib not available ({exc}); falling back to sequential execution.')
        return [fn(item) for item in items]

    n_jobs = cpu_count() if int(jobs) == -1 else int(jobs)
    try:
        return Parallel(
            n_jobs=n_jobs,
            prefer='threads' if prefer_threads else 'processes',
            batch_size=batch_size,
            verbose=verbose,
        )(delayed(fn)(item) for item in items)
    except Exception as exc:
        print(f'Parallel execution failed ({exc}); falling back to sequential execution.')
        return [fn(item) for item in items]

def grid_search_gpr_params(lon_array, lat_array, value_array, base_gpr_cfg, param_grid, cv_cfg, parallel_cfg=None):
    param_grid = dict(param_grid or {})
    if not param_grid:
        return pd.DataFrame()

    lon_array = np.asarray(lon_array, dtype=float)
    lat_array = np.asarray(lat_array, dtype=float)
    value_array = np.asarray(value_array, dtype=float)

    param_names = list(param_grid)
    combos = list(product(*param_grid.values()))
    if not combos:
        return pd.DataFrame()

    parallel_cfg = dict(parallel_cfg or {})
    combo_eval = partial(
        _evaluate_combo,
        param_names=param_names,
        lon_array=lon_array,
        lat_array=lat_array,
        value_array=value_array,
        base_gpr_cfg=base_gpr_cfg,
        cv_cfg=cv_cfg,
    )
    rows = _run_parallel_or_sequential(
        parallel_cfg.get('n_jobs', 1),
        combo_eval,
        combos,
        prefer_threads=parallel_cfg.get('prefer_threads', False),
        batch_size=parallel_cfg.get('batch_size', 'auto'),
        verbose=int(parallel_cfg.get('verbose', 0)),
    )

    results = pd.DataFrame(rows)
    if not results.empty:
        results = results.sort_values(
            'rmse_mean',
            key=lambda values: np.where(np.isfinite(values), values, np.inf),
        ).reset_index(drop=True)
    return results

def progressive_refinement_grid_search(lon_array, lat_array, value_array, base_gpr_cfg, search_cfg, cv_cfg):
    categorical_grid = dict(search_cfg.get('categorical_params') or {})
    numeric_cfg = dict(search_cfg.get('numeric_params') or {})
    parallel_cfg = dict(search_cfg.get('parallel') or {})
    top_k = int(search_cfg.get('top_k', 5))
    padding_fraction = float(search_cfg.get('padding_fraction', 0.25))
    max_iterations = int(search_cfg.get('max_iterations', 4))

    numeric_state = {}
    for name, meta in numeric_cfg.items():
        bounds = meta.get('bounds')
        if not bounds or len(bounds) != 2:
            raise ValueError(f'Missing 2-element bounds for numeric param {name!r}.')
        lower, upper = map(float, bounds)
        if lower >= upper:
            raise ValueError(f'Lower bound must be < upper bound for {name!r}.')
        initial_range = meta.get('initial_range', bounds)
        current_lower, current_upper = map(float, initial_range)
        current_lower = max(lower, current_lower)
        current_upper = min(upper, current_upper)
        if current_upper <= current_lower:
            current_lower, current_upper = lower, upper
        numeric_state[name] = {
            'bounds': (lower, upper),
            'current': (current_lower, current_upper),
            'points': max(3, int(meta.get('points', 10))),
            'min_span': float(meta.get('min_span', 1e-4)),
        }

    history_records = []
    combined_results = []
    best_overall_row = None
    best_overall_rmse = np.inf

    for iteration in range(1, max_iterations + 1):
        iteration_ranges = {name: tuple(state['current']) for name, state in numeric_state.items()}
        param_grid = {**categorical_grid}
        for name, state in numeric_state.items():
            low, high = iteration_ranges[name]
            param_grid[name] = np.linspace(low, high, state['points'])

        iter_results = grid_search_gpr_params(
            lon_array,
            lat_array,
            value_array,
            base_gpr_cfg,
            param_grid,
            cv_cfg,
            parallel_cfg=parallel_cfg,
        )

        if iter_results.empty:
            history_records.append({
                'iteration': iteration,
                'best_rmse': np.nan,
                **{f'{name}_min': rng[0] for name, rng in iteration_ranges.items()},
                **{f'{name}_max': rng[1] for name, rng in iteration_ranges.items()},
            })
            break

        iter_results = iter_results.assign(iteration=iteration)
        combined_results.append(iter_results)

        finite = iter_results[np.isfinite(iter_results['rmse_mean'])]
        if finite.empty:
            history_records.append({
                'iteration': iteration,
                'best_rmse': np.nan,
                **{f'{name}_min': rng[0] for name, rng in iteration_ranges.items()},
                **{f'{name}_max': rng[1] for name, rng in iteration_ranges.items()},
            })
            break

        best_row = finite.iloc[0]
        best_rmse = float(best_row['rmse_mean'])
        if best_rmse < best_overall_rmse:
            best_overall_rmse = best_rmse
            best_overall_row = best_row

        history_records.append({
            'iteration': iteration,
            'best_rmse': best_rmse,
            **{f'{name}_min': rng[0] for name, rng in iteration_ranges.items()},
            **{f'{name}_max': rng[1] for name, rng in iteration_ranges.items()},
        })

        if top_k <= 0:
            continue

        top_slice = finite.head(top_k)
        for name, state in numeric_state.items():
            lower_bound, upper_bound = state['bounds']
            span = iteration_ranges[name][1] - iteration_ranges[name][0]
            if span <= state['min_span']:
                continue

            top_min = float(top_slice[name].min())
            top_max = float(top_slice[name].max())
            if not np.isfinite(top_min) or not np.isfinite(top_max):
                continue

            padding = max(span * padding_fraction, state['min_span'])
            new_lower = max(lower_bound, top_min - padding)
            new_upper = min(upper_bound, top_max + padding)
            if new_upper - new_lower < state['min_span']:
                midpoint = 0.5 * (new_lower + new_upper)
                half_span = state['min_span'] / 2
                new_lower = max(lower_bound, midpoint - half_span)
                new_upper = min(upper_bound, midpoint + half_span)
            numeric_state[name]['current'] = (float(new_lower), float(new_upper))

    combined_df = pd.concat(combined_results, ignore_index=True) if combined_results else pd.DataFrame()
    if not combined_df.empty:
        combined_df = combined_df.sort_values(
            'rmse_mean',
            key=lambda values: np.where(np.isfinite(values), values, np.inf),
        ).reset_index(drop=True)

    history_df = pd.DataFrame(history_records)
    best_params = {}
    best_config = {}
    if best_overall_row is not None:
        for name in list(categorical_grid) + list(numeric_cfg):
            if name in best_overall_row:
                best_params[name] = best_overall_row[name]
        best_config = _apply_gpr_updates(base_gpr_cfg, best_params)
    return {
        'history': history_df,
        'results': combined_df,
        'best_params': best_params,
        'best_config': best_config,
    }

def _build_progressive_search_cfg(default_gpr, overrides=None):
    kernel_cfg = dict(default_gpr.get('kernel', {}))
    kernel_type = kernel_cfg.get('type', 'RBF')
    kernel_candidates = _unique_preserve([
        kernel_type,
        'RBF',
        'Matern',
        'RationalQuadratic',
    ])
    cfg = {
        'categorical_params': {
            'kernel.type': kernel_candidates,
        },
        'numeric_params': {
            'alpha': {
                'bounds': (0.0001, 1.0),
                'initial_range': (0.01, 0.5),
                'points': 6,
                'min_span': 5e-5,
            },
            'kernel.length_scale': {
                'bounds': (0.1, 25),
                'initial_range': (0.5, 10),
                'points': 6,
                'min_span': 5e-4,
            },
            'kernel.white_noise': {
                'bounds': (1e-5, 1.0),
                'initial_range': (1e-4, 0.3),
                'points': 6,
                'min_span': 1e-5,
            },
            'kernel.alpha': {
                'bounds': (0.1, 5.0),
                'initial_range': (0.2, 3.0),
                'points': 6,
                'min_span': 1e-3,
            },
            'kernel.constant': {
                'bounds': (0.1, 10.0),
                'initial_range': (0.5, 5.0),
                'points': 6,
                'min_span': 1e-3,
            },
        },
        'top_k': 6,
        'padding_fraction': 0.25,
        'max_iterations': 1,
        'parallel': {
            'n_jobs': -1,
            'prefer_threads': False,
            'batch_size': 'auto',
            'verbose': 0,
        },
    }

    overrides = overrides or {}
    for key, value in overrides.items():
        if key == 'parallel':
            cfg['parallel'].update(value)
        elif key in ('categorical_params', 'numeric_params'):
            cfg[key] = value
        else:
            cfg[key] = overrides[key]
    return cfg

base_gpr = dict(cfg.get('gpr', {}))
cv_cfg = dict(cfg.get('kfold', {}))
progressive_cfg = _build_progressive_search_cfg(base_gpr, cfg.get('grid_search'))

# Adjust parallelization dynamically without editing the helper.
# Example: progressive_cfg['parallel']['n_jobs'] = 8
progressive_results = progressive_refinement_grid_search(
    lon,
    lat,
    values,
    base_gpr,
    progressive_cfg,
    cv_cfg,
 )

history_df = progressive_results['history']
if not history_df.empty:
    print('Progressive refinement summary (ranges reflect the grid used each iteration):')
    display(history_df)
else:
    print('No history recorded; the refinement search did not complete any iterations.')

gpr_search_results = progressive_results['results']
print('Top candidate GPR settings across all iterations (sorted by mean RMSE):')
if not gpr_search_results.empty:
    display(gpr_search_results.head(10))
else:
    display(gpr_search_results)

best_gpr_params = progressive_results['best_params']
best_gpr_config = progressive_results['best_config']
if best_gpr_params:
    print('Best GPR hyperparameters based on progressive search:')
    best_gpr_params
else:
    print('No successful parameter combinations were found. Inspect gpr_search_results for diagnostics.')

